In [6]:
import numpy as np, pandas as pd
items = pd.read_excel('C:/medicalAI/Pycham/week3/Data_processing/recommend/items_user.xlsx')
history = pd.read_excel('C:/medicalAI/Pycham/week3/Data_processing/recommend/items_user.xlsx', sheet_name='purchase_history')
print(items.to_string(index=False))
print('\npurchase history:')
print(history.to_string(index=False))

item_id color  weight  item_size   price
  item1 black     100         90  150000
  item2  gray     200        105   25000
  item3 black     220         90  101000
  item4 white     100         90   10000
  item5 black     230        105  105000
  item6 black     250        105 1500000
  item7 white     110        105 1000000
  item8  gray     250         90 1100000
  item9 black     300         95  205000
 item10 black     120        105  350000

purchase history:
 seq mem_id item_id
   1  user1   item1
   2  user1   item3
   3  user1   item6


In [14]:
profile = pd.get_dummies(items, columns=['color'])
#number grouping
profile['light'] = items['weight'] <= 200
profile['heavy'] = items['weight'] > 200
profile['small'] = items['item_size'] <= 95
profile['big'] = items['item_size'] > 95
profile['low'] = items['price'] <=50000
profile['medium'] = (items['price'] > 50000) & (items['price']) <= 500000
profile['high'] = items['price'] >500000

#drop
profile = profile.drop(columns=['item_id', 'weight', 'item_size', 'price']).astype(int)
profile.index = items['item_id']
profile


,color_black,color_gray,color_white,light,heavy,small,big,low,medium,high
item_id,,,,,,,,,,
item1,1,0,0,1,0,1,0,0,1,0
item2,0,1,0,1,0,0,1,1,1,0
item3,1,0,0,0,1,1,0,0,1,0
item4,0,0,1,1,0,1,0,1,1,0
item5,1,0,0,0,1,0,1,0,1,0
item6,1,0,0,0,1,0,1,0,1,1
item7,0,0,1,1,0,0,1,0,1,1
item8,0,1,0,0,1,1,0,0,1,1
item9,1,0,0,0,1,1,0,0,1,0


In [15]:
# customer prefer
user_items = history[history['mem_id']== 'user1']['item_id']
print('customer purchase item', user_items))
user_profile = profile.loc[user_items].mean()
user_profile


customer purchase item 0    item1
1    item3
2    item6
Name: item_id, dtype: str


color_black    1.000000
color_gray     0.000000
color_white    0.000000
light          0.333333
heavy          0.666667
small          0.666667
big            0.333333
low            0.000000
medium         1.000000
high           0.333333
dtype: float64

In [10]:
#similarity
def cos_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a)* np.linalg.norm(b))
unseen = profile.drop(index=user_items) #unpurchased item
scores = unseen.apply(lambda row: cos_sim(user_profile.values, row.values), axis=1)
scores = scores.sort_values(ascending=False).round(3)
print(scores)

item_id
item9     0.928
item5     0.836
item10    0.743
item8     0.664
item7     0.498
item4     0.498
item2     0.415
dtype: float64
